# 🔧 DATA PREPROCESSING PIPELINE

This notebook transforms raw demand data into ML-ready features:
1. Load and explore raw data
2. Create continuous time series (fill missing dates)
3. Add temporal features with cyclical encoding
4. Create lag features (1d, 7d, 14d, 30d)
5. Create rolling features (mean, std, max, min)
6. Calculate product-level statistics
7. Handle missing values
8. Temporal train/test split
9. Save processed data

**WITH COMPREHENSIVE VISUALIZATIONS AT EACH STEP**

---

## STEP 1: LOAD RAW DATA

In [ ]:
# ===================================================================
# LOAD AND EXPLORE RAW DATA
# ===================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Loading raw data...")
df = pd.read_csv('../data/raw/products_for_ts_grouped.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"\n📊 RAW DATA SHAPE: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Total days: {(df['date'].max() - df['date'].min()).days + 1}")
print(f"Unique products: {df['id_produit'].nunique()}")
print(f"Total demand: {df['quantite_demande'].sum():,.0f} units")
print(f"\nColumns: {list(df.columns)}")

df.head(10)

In [ ]:
# ===================================================================
# VISUALIZATION 1: RAW DATA OVERVIEW
# ===================================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Daily total demand
daily_demand = df.groupby('date')['quantite_demande'].sum()
axes[0, 0].plot(daily_demand.index, daily_demand.values, linewidth=1.5)
axes[0, 0].set_title('Daily Total Demand Over Time', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Total Units')
axes[0, 0].grid(True, alpha=0.3)

# 2. Demand distribution (log scale)
axes[0, 1].hist(df['quantite_demande'], bins=100, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Demand Distribution (All Products)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Quantity per Day')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_yscale('log')
axes[0, 1].grid(True, alpha=0.3)

# 3. Products per day
products_per_day = df.groupby('date')['id_produit'].nunique()
axes[1, 0].plot(products_per_day.index, products_per_day.values, linewidth=1.5, color='green')
axes[1, 0].set_title('Number of Active Products per Day', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Product Count')
axes[1, 0].grid(True, alpha=0.3)

# 4. Top 10 products by total demand
top_products = df.groupby('id_produit')['quantite_demande'].sum().nlargest(10)
axes[1, 1].barh(range(len(top_products)), top_products.values, alpha=0.7)
axes[1, 1].set_yticks(range(len(top_products)))
axes[1, 1].set_yticklabels([f'Product {pid}' for pid in top_products.index])
axes[1, 1].set_title('Top 10 Products by Total Demand', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Total Units')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n✓ Raw data visualizations complete")

## STEP 2: CREATE CONTINUOUS TIME SERIES

In [ ]:
# ===================================================================
# FILL MISSING DATES WITH ZEROS
# ===================================================================

print("\n" + "="*60)
print("STEP 2: Creating continuous time series...")
print("="*60)

def create_continuous_timeseries(df):
    """
    Fill missing dates for each product with 0 demand
    This is CRITICAL for time series forecasting
    """
    date_min = df['date'].min()
    date_max = df['date'].max()
    all_dates = pd.date_range(start=date_min, end=date_max, freq='D')
    all_products = df['id_produit'].unique()
    
    print(f"Date range: {date_min.date()} → {date_max.date()}")
    print(f"Total dates: {len(all_dates)}")
    print(f"Total products: {len(all_products)}")
    print(f"Expected rows: {len(all_dates) * len(all_products):,}")
    
    # Create complete grid
    idx = pd.MultiIndex.from_product([all_products, all_dates], names=['id_produit', 'date'])
    df_complete = df.set_index(['id_produit', 'date']).reindex(idx).reset_index()
    
    # Fill demand with 0
    df_complete['quantite_demande'] = df_complete['quantite_demande'].fillna(0)
    
    # Forward-fill static attributes
    static_cols = [c for c in df_complete.columns if c not in ['id_produit', 'date', 'quantite_demande']]
    if static_cols:
        df_complete[static_cols] = df_complete.groupby('id_produit')[static_cols].ffill().bfill()
    
    # Sort
    df_complete = df_complete.sort_values(['id_produit', 'date']).reset_index(drop=True)
    
    print(f"\n✓ Complete dataset shape: {df_complete.shape}")
    print(f"Rows with demand = 0: {(df_complete['quantite_demande'] == 0).sum():,}")
    print(f"Rows with demand > 0: {(df_complete['quantite_demande'] > 0).sum():,}")
    
    return df_complete

df_complete = create_continuous_timeseries(df)
df_complete.head(20)

In [ ]:
# ===================================================================
# VISUALIZATION 2: CONTINUOUS TIME SERIES IMPACT
# ===================================================================

# Pick a sample product to visualize
sample_product = df['id_produit'].value_counts().index[0]

fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Before: sparse data
sparse_data = df[df['id_produit'] == sample_product].sort_values('date')
axes[0].plot(sparse_data['date'], sparse_data['quantite_demande'], marker='o', linewidth=1.5)
axes[0].set_title(f'BEFORE: Sparse Data (Product {sample_product}) - Only {len(sparse_data)} days recorded', 
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Demand')
axes[0].grid(True, alpha=0.3)

# After: continuous with zeros
continuous_data = df_complete[df_complete['id_produit'] == sample_product].sort_values('date')
axes[1].plot(continuous_data['date'], continuous_data['quantite_demande'], linewidth=1.5)
axes[1].set_title(f'AFTER: Continuous Time Series - All {len(continuous_data)} days filled', 
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Demand')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Visualization shows importance of filling missing dates")
print(f"  Missing dates filled: {len(continuous_data) - len(sparse_data):,}")

## STEP 3: ADD TEMPORAL FEATURES

In [ ]:
# ===================================================================
# TEMPORAL FEATURES WITH CYCLICAL ENCODING
# ===================================================================

print("\n" + "="*60)
print("STEP 3: Adding temporal features...")
print("="*60)

def add_temporal_features(df):
    """
    Add 13 temporal features including cyclical encoding
    """
    df = df.copy()
    
    # Basic temporal features
    df['day_of_week'] = df['date'].dt.dayofweek  # 0=Monday, 6=Sunday
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    
    # Binary features
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_month_start'] = (df['day'] <= 7).astype(int)
    df['is_month_end'] = (df['day'] >= 24).astype(int)
    
    # Cyclical encoding (CRITICAL for ML)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    # Day of year (for long-term seasonality)
    df['day_of_year'] = df['date'].dt.dayofyear
    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    
    print("✓ Added 13 temporal features:")
    temporal_features = ['day_of_week', 'month', 'day', 'week_of_year', 'day_of_year',
                        'is_weekend', 'is_month_start', 'is_month_end',
                        'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos']
    for feat in temporal_features:
        print(f"  - {feat}")
    
    return df

df_complete = add_temporal_features(df_complete)
print(f"\nDataset shape: {df_complete.shape}")

In [ ]:
# ===================================================================
# VISUALIZATION 3: TEMPORAL PATTERNS
# ===================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Demand by day of week
dow_demand = df_complete.groupby('day_of_week')['quantite_demande'].mean()
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[0, 0].bar(range(7), dow_demand.values, alpha=0.7, color='steelblue')
axes[0, 0].set_xticks(range(7))
axes[0, 0].set_xticklabels(dow_labels)
axes[0, 0].set_title('Average Demand by Day of Week', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Avg Units')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Demand by month
month_demand = df_complete.groupby('month')['quantite_demande'].mean()
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[0, 1].bar(range(1, 13), month_demand.values, alpha=0.7, color='coral')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].set_xticklabels(month_labels, rotation=45)
axes[0, 1].set_title('Average Demand by Month', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Avg Units')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Weekend vs Weekday
weekend_demand = df_complete.groupby('is_weekend')['quantite_demande'].mean()
axes[0, 2].bar(['Weekday', 'Weekend'], weekend_demand.values, alpha=0.7, color=['green', 'orange'])
axes[0, 2].set_title('Weekday vs Weekend Demand', fontsize=11, fontweight='bold')
axes[0, 2].set_ylabel('Avg Units')
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. Cyclical encoding: Month
axes[1, 0].scatter(df_complete['month_sin'], df_complete['month_cos'], alpha=0.1, s=1)
axes[1, 0].set_title('Cyclical Encoding: Month (sin/cos)', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('month_sin')
axes[1, 0].set_ylabel('month_cos')
axes[1, 0].grid(True, alpha=0.3)

# 5. Cyclical encoding: Day of week
axes[1, 1].scatter(df_complete['dow_sin'], df_complete['dow_cos'], alpha=0.1, s=1)
axes[1, 1].set_title('Cyclical Encoding: Day of Week (sin/cos)', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('dow_sin')
axes[1, 1].set_ylabel('dow_cos')
axes[1, 1].grid(True, alpha=0.3)

# 6. Month start/end impact
month_period_demand = df_complete.groupby(['is_month_start', 'is_month_end'])['quantite_demande'].mean().unstack(fill_value=0)
im = axes[1, 2].imshow(month_period_demand.values, cmap='YlOrRd', aspect='auto')
axes[1, 2].set_xticks([0, 1])
axes[1, 2].set_xticklabels(['Mid-month', 'Month-end'])
axes[1, 2].set_yticks([0, 1])
axes[1, 2].set_yticklabels(['Mid-month', 'Month-start'])
axes[1, 2].set_title('Demand Heatmap: Month Periods', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=axes[1, 2])

plt.tight_layout()
plt.show()

print("\n✓ Temporal pattern visualizations complete")

## STEP 4: CREATE LAG FEATURES

In [ ]:
# ===================================================================
# LAG FEATURES (CRITICAL FOR TIME SERIES)
# ===================================================================

print("\n" + "="*60)
print("STEP 4: Creating lag features...")
print("="*60)

def create_lag_features(df, lags=[1, 7, 14, 30]):
    """
    Create lag features for each product
    """
    df = df.copy()
    
    for lag in lags:
        df[f'lag_{lag}d'] = df.groupby('id_produit')['quantite_demande'].shift(lag)
        print(f"  ✓ lag_{lag}d created")
    
    return df

df_complete = create_lag_features(df_complete)
print(f"\nDataset shape: {df_complete.shape}")

In [ ]:
# ===================================================================
# VISUALIZATION 4: LAG FEATURE CORRELATIONS
# ===================================================================

# Calculate correlations for lag features
lag_cols = ['quantite_demande', 'lag_1d', 'lag_7d', 'lag_14d', 'lag_30d']
sample_data = df_complete[lag_cols].dropna().sample(min(50000, len(df_complete)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Correlation heatmap
corr_matrix = sample_data.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, ax=axes[0], cbar_kws={'label': 'Correlation'})
axes[0].set_title('Lag Feature Correlations', fontsize=12, fontweight='bold')

# 2. Scatter: lag_7d vs target
axes[1].scatter(sample_data['lag_7d'], sample_data['quantite_demande'], alpha=0.3, s=5)
axes[1].plot([0, sample_data['lag_7d'].max()], [0, sample_data['lag_7d'].max()], 
             'r--', linewidth=2, label='Perfect correlation')
axes[1].set_xlabel('Lag 7d (Last Week)')
axes[1].set_ylabel('Current Demand')
axes[1].set_title('Lag 7d vs Current Demand', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Lag feature analysis complete")
print(f"Correlation with target:")
for col in ['lag_1d', 'lag_7d', 'lag_14d', 'lag_30d']:
    corr = corr_matrix.loc['quantite_demande', col]
    print(f"  {col}: {corr:.3f}")

## STEP 5: CREATE ROLLING FEATURES

In [ ]:
# ===================================================================
# ROLLING WINDOW FEATURES
# ===================================================================

print("\n" + "="*60)
print("STEP 5: Creating rolling features...")
print("="*60)

def create_rolling_features(df, windows=[7, 14, 30]):
    """
    Create rolling statistics for each product
    """
    df = df.copy()
    
    for window in windows:
        # Rolling mean
        df[f'rolling_mean_{window}d'] = df.groupby('id_produit')['quantite_demande'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )
        
        # Rolling std
        df[f'rolling_std_{window}d'] = df.groupby('id_produit')['quantite_demande'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).std()
        )
        
        # Rolling max
        df[f'rolling_max_{window}d'] = df.groupby('id_produit')['quantite_demande'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).max()
        )
        
        # Rolling min
        df[f'rolling_min_{window}d'] = df.groupby('id_produit')['quantite_demande'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).min()
        )
        
        print(f"  ✓ Rolling features for {window}d window created (mean, std, max, min)")
    
    return df

df_complete = create_rolling_features(df_complete)
print(f"\nDataset shape: {df_complete.shape}")
print(f"Total features created: {df_complete.shape[1]}")

In [ ]:
# ===================================================================
# VISUALIZATION 5: ROLLING FEATURES EXAMPLE
# ===================================================================

# Pick a high-demand product for visualization
top_product = df.groupby('id_produit')['quantite_demande'].sum().nlargest(1).index[0]
product_data = df_complete[df_complete['id_produit'] == top_product].sort_values('date')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# 1. Rolling means
axes[0].plot(product_data['date'], product_data['quantite_demande'], 
             label='Actual Demand', linewidth=2, alpha=0.7)
axes[0].plot(product_data['date'], product_data['rolling_mean_7d'], 
             label='7-day MA', linewidth=1.5, linestyle='--')
axes[0].plot(product_data['date'], product_data['rolling_mean_14d'], 
             label='14-day MA', linewidth=1.5, linestyle='--')
axes[0].plot(product_data['date'], product_data['rolling_mean_30d'], 
             label='30-day MA', linewidth=1.5, linestyle='--')
axes[0].set_title(f'Rolling Means for Product {top_product}', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Quantity')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. Rolling std (volatility)
axes[1].plot(product_data['date'], product_data['rolling_std_7d'], 
             label='7-day Std', linewidth=1.5)
axes[1].plot(product_data['date'], product_data['rolling_std_14d'], 
             label='14-day Std', linewidth=1.5)
axes[1].plot(product_data['date'], product_data['rolling_std_30d'], 
             label='30-day Std', linewidth=1.5)
axes[1].set_title('Rolling Standard Deviation (Volatility)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Std Deviation')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Rolling feature visualizations complete")

## STEP 6: CALCULATE PRODUCT-LEVEL STATISTICS

In [ ]:
# ===================================================================
# PRODUCT-LEVEL STATISTICS
# ===================================================================

print("\n" + "="*60)
print("STEP 6: Calculating product-level statistics...")
print("="*60)

def calculate_product_stats(df):
    """
    Calculate product-level statistics
    """
    df = df.copy()
    
    product_stats = df.groupby('id_produit').agg({
        'quantite_demande': ['count', 'mean', 'std', 'sum']
    })
    
    product_stats.columns = ['order_frequency', 'avg_quantity', 'std_quantity', 'total_quantity']
    product_stats = product_stats.reset_index()
    
    # Coefficient of variation (volatility measure)
    product_stats['cv'] = product_stats['std_quantity'] / (product_stats['avg_quantity'] + 1)
    
    # Merge back to main dataframe
    df = df.merge(product_stats[['id_produit', 'order_frequency', 'cv', 'avg_quantity']], 
                  on='id_produit', how='left')
    
    print(f"✓ Added 3 product-level features:")
    print(f"  - order_frequency (total orders)")
    print(f"  - cv (coefficient of variation)")
    print(f"  - avg_quantity (average demand)")
    
    return df, product_stats

df_complete, product_stats = calculate_product_stats(df_complete)
print(f"\nDataset shape: {df_complete.shape}")

In [ ]:
# ===================================================================
# VISUALIZATION 6: PRODUCT STATISTICS
# ===================================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Order frequency distribution
axes[0, 0].hist(product_stats['order_frequency'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Order Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Number of Orders')
axes[0, 0].set_ylabel('Number of Products')
axes[0, 0].set_yscale('log')
axes[0, 0].grid(True, alpha=0.3)

# 2. Average quantity distribution
axes[0, 1].hist(product_stats['avg_quantity'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of Average Quantity', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Average Demand')
axes[0, 1].set_ylabel('Number of Products')
axes[0, 1].set_yscale('log')
axes[0, 1].grid(True, alpha=0.3)

# 3. Coefficient of variation
axes[1, 0].hist(product_stats['cv'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('Distribution of CV (Volatility)', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Coefficient of Variation')
axes[1, 0].set_ylabel('Number of Products')
axes[1, 0].grid(True, alpha=0.3)

# 4. Frequency vs Volume scatter
axes[1, 1].scatter(product_stats['order_frequency'], product_stats['avg_quantity'], 
                   alpha=0.5, s=30, c=product_stats['cv'], cmap='viridis')
axes[1, 1].set_title('Order Frequency vs Average Quantity', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Order Frequency')
axes[1, 1].set_ylabel('Average Quantity')
axes[1, 1].set_xscale('log')
axes[1, 1].set_yscale('log')
cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
cbar.set_label('CV (Volatility)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Product statistics visualizations complete")

## STEP 7: HANDLE MISSING VALUES

In [ ]:
# ===================================================================
# MISSING VALUE HANDLING
# ===================================================================

print("\n" + "="*60)
print("STEP 7: Handling missing values...")
print("="*60)

print("\nMissing values before handling:")
missing_counts = df_complete.isnull().sum()
missing_features = missing_counts[missing_counts > 0]
if len(missing_features) > 0:
    for col, count in missing_features.items():
        pct = (count / len(df_complete)) * 100
        print(f"  {col:30s}: {count:8,} ({pct:.2f}%)")
else:
    print("  No missing values found!")

# Fill remaining NaN in rolling features with 0
rolling_cols = [c for c in df_complete.columns if 'rolling_' in c]
for col in rolling_cols:
    df_complete[col] = df_complete[col].fillna(0)

print("\n✓ Missing values handled")
print(f"Dataset shape: {df_complete.shape}")

In [ ]:
# ===================================================================
# VISUALIZATION 7: MISSING VALUE PATTERNS
# ===================================================================

# Check lag features for missing patterns (expected at start of time series)
lag_cols = [c for c in df_complete.columns if 'lag_' in c]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Missing values heatmap (sample)
sample_product = df_complete['id_produit'].iloc[0]
sample_data = df_complete[df_complete['id_produit'] == sample_product].head(50)
missing_matrix = sample_data[['quantite_demande'] + lag_cols].isnull().astype(int)
sns.heatmap(missing_matrix.T, cmap='RdYlGn_r', cbar=False, ax=axes[0])
axes[0].set_title(f'Missing Values Pattern (Product {sample_product}, first 50 days)', 
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Day Index')
axes[0].set_ylabel('Features')

# 2. Missing value counts per feature
missing_counts = df_complete[lag_cols + rolling_cols].isnull().sum()
if missing_counts.sum() > 0:
    missing_counts.plot(kind='bar', ax=axes[1], alpha=0.7)
    axes[1].set_title('Missing Values per Feature', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Count')
    axes[1].set_xlabel('Feature')
    axes[1].tick_params(axis='x', rotation=45)
else:
    axes[1].text(0.5, 0.5, 'No Missing Values!', 
                 ha='center', va='center', fontsize=16, fontweight='bold')
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Missing value analysis complete")

## STEP 8: TEMPORAL TRAIN/TEST SPLIT

In [ ]:
# ===================================================================
# TEMPORAL TRAIN/TEST SPLIT
# ===================================================================

print("\n" + "="*60)
print("STEP 8: Creating temporal train/test split...")
print("="*60)

# Use last 7 days for testing
max_date = df_complete['date'].max()
test_start_date = max_date - timedelta(days=6)  # Last 7 days

train = df_complete[df_complete['date'] < test_start_date].copy()
test = df_complete[df_complete['date'] >= test_start_date].copy()

print(f"\nSplit date: {test_start_date.date()}")
print(f"\nTrain set:")
print(f"  Dates: {train['date'].min().date()} → {train['date'].max().date()}")
print(f"  Rows: {len(train):,}")
print(f"  Products: {train['id_produit'].nunique()}")
print(f"  Total demand: {train['quantite_demande'].sum():,.0f}")

print(f"\nTest set:")
print(f"  Dates: {test['date'].min().date()} → {test['date'].max().date()}")
print(f"  Rows: {len(test):,}")
print(f"  Products: {test['id_produit'].nunique()}")
print(f"  Total demand: {test['quantite_demande'].sum():,.0f}")

print(f"\n✓ Train/test split complete")

In [ ]:
# ===================================================================
# VISUALIZATION 8: TRAIN/TEST SPLIT
# ===================================================================

fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# 1. Daily demand with split line
train_daily = train.groupby('date')['quantite_demande'].sum()
test_daily = test.groupby('date')['quantite_demande'].sum()

axes[0].plot(train_daily.index, train_daily.values, label='Train', linewidth=1.5, color='blue')
axes[0].plot(test_daily.index, test_daily.values, label='Test', linewidth=1.5, color='red')
axes[0].axvline(x=test_start_date, color='green', linestyle='--', linewidth=2, label='Split Date')
axes[0].set_title('Train/Test Split: Daily Total Demand', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Total Units')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Feature distributions comparison
feature_to_compare = 'lag_7d'
if feature_to_compare in train.columns:
    train_sample = train[feature_to_compare].dropna().sample(min(10000, len(train)), random_state=42)
    test_sample = test[feature_to_compare].dropna().sample(min(10000, len(test)), random_state=42)
    
    axes[1].hist(train_sample, bins=50, alpha=0.6, label='Train', edgecolor='black')
    axes[1].hist(test_sample, bins=50, alpha=0.6, label='Test', edgecolor='black')
    axes[1].set_title(f'Feature Distribution Comparison: {feature_to_compare}', 
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Value')
    axes[1].set_ylabel('Frequency')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Train/test split visualizations complete")

## STEP 9: SAVE PROCESSED DATA

In [ ]:
# ===================================================================
# SAVE PROCESSED DATASETS
# ===================================================================

print("\n" + "="*60)
print("STEP 9: Saving processed data...")
print("="*60)

# Save complete dataset
df_complete.to_csv('../data/processed/features_complete.csv', index=False)
print(f"✓ Saved: features_complete.csv ({len(df_complete):,} rows, {len(df_complete.columns)} columns)")

# Save train set
train.to_csv('../data/processed/train.csv', index=False)
print(f"✓ Saved: train.csv ({len(train):,} rows)")

# Save test set
test.to_csv('../data/processed/test.csv', index=False)
print(f"✓ Saved: test.csv ({len(test):,} rows)")

# Save product statistics
product_stats.to_csv('../data/processed/product_stats.csv', index=False)
print(f"✓ Saved: product_stats.csv ({len(product_stats)} products)")

print("\n" + "="*60)
print("🎉 PREPROCESSING COMPLETE!")
print("="*60)
print(f"\nTotal features created: {len(df_complete.columns)}")
print(f"\nFeature categories:")
print(f"  - TARGET: 1 (quantite_demande)")
print(f"  - IDs: 2 (id_produit, date)")
print(f"  - TEMPORAL: 14 (including cyclical encoding)")
print(f"  - LAG: 4 (1d, 7d, 14d, 30d)")
print(f"  - ROLLING: 12 (mean/std/max/min for 7/14/30d)")
print(f"  - PRODUCT STATS: 3 (order_frequency, cv, avg_quantity)")
print(f"  - ATTRIBUTES: {len([c for c in df_complete.columns if c not in ['id_produit', 'date', 'quantite_demande'] and not any(x in c for x in ['lag_', 'rolling_', 'month', 'dow', 'day', 'week', 'is_', 'order_', 'cv', 'avg_'])])}")
print(f"\nNext step: Run notebook 03 for model training!")

In [ ]:
# ===================================================================
# FINAL VISUALIZATION: FEATURE SUMMARY
# ===================================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Feature count by category
feature_categories = {
    'Lag Features': 4,
    'Rolling Features': 12,
    'Temporal Features': 14,
    'Product Stats': 3,
    'Attributes': len([c for c in df_complete.columns if c not in ['id_produit', 'date', 'quantite_demande'] and not any(x in c for x in ['lag_', 'rolling_', 'month', 'dow', 'day', 'week', 'is_', 'order_', 'cv', 'avg_'])])
}

axes[0].bar(range(len(feature_categories)), list(feature_categories.values()), alpha=0.7)
axes[0].set_xticks(range(len(feature_categories)))
axes[0].set_xticklabels(list(feature_categories.keys()), rotation=45, ha='right')
axes[0].set_title('Engineered Features by Category', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Feature Count')
axes[0].grid(True, alpha=0.3, axis='y')

# 2. Dataset size comparison
dataset_sizes = {
    'Raw Data': len(df),
    'Complete\n(with zeros)': len(df_complete),
    'Train Set': len(train),
    'Test Set': len(test)
}

axes[1].bar(range(len(dataset_sizes)), list(dataset_sizes.values()), 
            alpha=0.7, color=['blue', 'green', 'orange', 'red'])
axes[1].set_xticks(range(len(dataset_sizes)))
axes[1].set_xticklabels(list(dataset_sizes.keys()))
axes[1].set_title('Dataset Sizes', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Row Count')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (name, value) in enumerate(dataset_sizes.items()):
    axes[1].text(i, value, f'{value:,}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n✓ All preprocessing visualizations complete!")